# Notebook 07: Reconnection Scaling and Dimensionless Bridge

This notebook translates the geometric current-sheet fragmentation model into a **dimensionless, physics-facing language**.

Notebook 06 mapped the regime structure in raw parameters:

- \(\nu\): diffusion / smoothing
- \(\beta\): activity strength near the 45° gate
- \(\gamma\): damping of the transverse field

Notebook 07 asks:

- can the fragmentation observables be organized by **dimensionless control ratios**?
- do the regimes collapse more cleanly in timescale-ratio variables than in raw parameter coordinates?

We define three effective control numbers:

\[
\Pi_1 = \frac{\beta}{\nu}
\]

\[
\Pi_2 = \frac{\beta}{\gamma}
\]

\[
\Pi_3 = \beta \frac{\delta^2}{\nu}
\]

Interpretation:

- \(\Pi_1\): activity-to-diffusion ratio
- \(\Pi_2\): activity-to-damping ratio
- \(\Pi_3\): thickness-aware drive-to-smoothing ratio

This is a **dimensionless bridge**, not a claim of exact MHD equivalence.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import deque

THRESHOLD = 1 / np.sqrt(2)

nx, ny = 260, 200
x = np.linspace(-6.0, 6.0, nx)
y = np.linspace(-3.0, 3.0, ny)
X, Y = np.meshgrid(x, y)

dx = x[1] - x[0]
dy = y[1] - y[0]

print("45° threshold =", THRESHOLD)
print("grid shape =", X.shape)

## 1. Shared helpers

We reuse the current-sheet model and the pseudo-time dynamics in a slightly lighter form so the scaling notebook stays practical.

In [ ]:
def normalize_field(Bx, By):
    mag = np.sqrt(Bx**2 + By**2)
    mag = np.where(mag == 0, 1.0, mag)
    return Bx / mag, By / mag

def multimode_sheet_perturbation(X, Y, amps, ks, phases, sigma=0.38):
    out = np.zeros_like(X)
    envelope = np.exp(-(Y**2) / (sigma**2))
    for a, k, p in zip(amps, ks, phases):
        out += a * np.sin(k * X + p)
    return envelope * out

def harris_target(X, Y, delta=0.35):
    return np.tanh(Y / delta)

def initial_field(X, Y, delta=0.35, base_eps=0.06, frag_amp=0.28, sigma=0.38, rng=None):
    if rng is None:
        rng = np.random.default_rng(0)

    Bx = np.tanh(Y / delta)

    By_background = base_eps * np.sin(1.2 * X) * np.exp(-(Y**2) / (1.0**2))

    ks = [1.0, 2.0, 3.6, 5.0]
    phases = rng.uniform(0, 2 * np.pi, size=len(ks))
    amps = frag_amp * np.array([1.0, 0.75, 0.55, 0.35])
    amps = amps * (1.0 + 0.15 * rng.standard_normal(len(ks)))

    By_fragment = multimode_sheet_perturbation(X, Y, amps, ks, phases, sigma=sigma)

    By = By_background + By_fragment
    return Bx, By

def random_seed_pattern(X, Y, base_amp=0.28, sigma=0.38, rng=None):
    if rng is None:
        rng = np.random.default_rng(0)
    ks = [1.0, 2.0, 3.6, 5.0]
    phases = rng.uniform(0, 2 * np.pi, size=len(ks))
    amps = base_amp * np.array([1.0, 0.75, 0.55, 0.35])
    amps = amps * (1.0 + 0.15 * rng.standard_normal(len(ks)))
    return multimode_sheet_perturbation(X, Y, amps, ks, phases, sigma=sigma)

def laplacian(F, dx, dy):
    return (
        (np.roll(F, 1, axis=0) - 2 * F + np.roll(F, -1, axis=0)) / dy**2 +
        (np.roll(F, 1, axis=1) - 2 * F + np.roll(F, -1, axis=1)) / dx**2
    )

def cross_sheet_cosine(Bx_hat, By_hat, shift=3):
    Bx_up = np.roll(Bx_hat, -shift, axis=0)
    By_up = np.roll(By_hat, -shift, axis=0)

    Bx_dn = np.roll(Bx_hat, shift, axis=0)
    By_dn = np.roll(By_hat, shift, axis=0)

    cos_map = Bx_up * Bx_dn + By_up * By_dn
    cos_map[:shift, :] = np.nan
    cos_map[-shift:, :] = np.nan
    return cos_map

def failed_mask(cos_map, threshold=THRESHOLD):
    return cos_map < threshold

def central_band_mask(mask, Y, half_width=0.8):
    return mask & (np.abs(Y) < half_width)

def connected_components(mask):
    visited = np.zeros_like(mask, dtype=bool)
    rows, cols = mask.shape
    count = 0

    for i in range(rows):
        for j in range(cols):
            if mask[i, j] and not visited[i, j]:
                count += 1
                q = deque([(i, j)])
                visited[i, j] = True

                while q:
                    r, c = q.popleft()
                    for rr, cc in [(r-1, c), (r+1, c), (r, c-1), (r, c+1)]:
                        if 0 <= rr < rows and 0 <= cc < cols:
                            if mask[rr, cc] and not visited[rr, cc]:
                                visited[rr, cc] = True
                                q.append((rr, cc))
    return count

def estimate_failed_width_vs_x(cos_map, y, threshold=THRESHOLD):
    widths = np.full(cos_map.shape[1], np.nan)
    for j in range(cos_map.shape[1]):
        col = cos_map[:, j]
        bad = np.isfinite(col) & (col < threshold)
        if np.any(bad):
            y_bad = y[bad]
            widths[j] = y_bad.max() - y_bad.min()
    return widths

def constraint_energy(cos_map, threshold=THRESHOLD):
    return float(np.nanmean(np.maximum(0.0, threshold - cos_map)))

def one_step(Bx, By, Bx_target, seed_pattern, rng, dt, dx, dy, nu, alpha, beta, gamma, noise_amp=0.0, noise_sigma=0.38):
    Bx_hat, By_hat = normalize_field(Bx, By)
    cos_map = cross_sheet_cosine(Bx_hat, By_hat, shift=3)
    act = np.maximum(0.0, THRESHOLD - cos_map)
    act = np.nan_to_num(act, nan=0.0)

    sheet_noise = noise_amp * rng.standard_normal(X.shape) * np.exp(-(Y**2) / (noise_sigma**2))

    Bx_new = Bx + dt * (
        nu * laplacian(Bx, dx, dy)
        - alpha * (Bx - Bx_target)
    )

    By_new = By + dt * (
        nu * laplacian(By, dx, dy)
        + beta * act * seed_pattern
        - gamma * By
        + sheet_noise
    )

    Bx_new[0, :] = Bx_target[0, :]
    Bx_new[-1, :] = Bx_target[-1, :]
    By_new[0, :] = 0.0
    By_new[-1, :] = 0.0

    return Bx_new, By_new

## 2. Parameter choices

We use a compact sweep over \(\nu\), \(\beta\), \(\gamma\), and a small thickness sweep \(\delta\) to test thickness-aware scaling.

In [ ]:
frag_amp = 0.28
sigma_seed = 0.38

alpha = 0.25
dt = 0.0018
noise_amp = 0.004
noise_sigma = 0.38
n_steps = 120

nu_vals = [0.0008, 0.0014, 0.0022]
beta_vals = [0.5, 1.0, 1.5]
gamma_vals = [0.10, 0.18, 0.28]
delta_vals = [0.25, 0.35, 0.50]

n_runs_per_point = 3

print("nu values   =", nu_vals)
print("beta values =", beta_vals)
print("gamma values=", gamma_vals)
print("delta values=", delta_vals)
print("runs per point =", n_runs_per_point)

## 3. Parameter-point runner

At each parameter point we run a small ensemble and record final-state observables.

In [ ]:
def run_parameter_point(delta, nu, beta, gamma, n_runs=3):
    final_pocket_counts = []
    final_width_stds = []
    final_constraint_energies = []
    final_failed_fractions = []
    critical_steps = []
    example_snapshot = None

    for run_id in range(n_runs):
        rng = np.random.default_rng(90000 + run_id)

        Bx_t, By_t = initial_field(
            X, Y, delta=delta, base_eps=0.06, frag_amp=frag_amp, sigma=sigma_seed, rng=rng
        )
        Bx_target = harris_target(X, Y, delta=delta)
        seed_pattern = random_seed_pattern(X, Y, base_amp=frag_amp, sigma=sigma_seed, rng=rng)

        failed_fraction_trace = []

        for step in range(n_steps + 1):
            Bx_hat_t, By_hat_t = normalize_field(Bx_t, By_t)
            cos_map_t = cross_sheet_cosine(Bx_hat_t, By_hat_t, shift=3)
            mask_t = failed_mask(cos_map_t)
            central_t = central_band_mask(mask_t, Y, half_width=0.8)

            failed_fraction_trace.append(float(np.nanmean(mask_t)))

            if step == n_steps:
                pocket_count = connected_components(np.nan_to_num(central_t, nan=False).astype(bool))
                widths_t = estimate_failed_width_vs_x(cos_map_t, y, threshold=THRESHOLD)

                final_pocket_counts.append(float(pocket_count))
                final_width_stds.append(float(np.nanstd(widths_t)) if np.any(np.isfinite(widths_t)) else np.nan)
                final_constraint_energies.append(constraint_energy(cos_map_t))
                final_failed_fractions.append(float(np.nanmean(mask_t)))

                if example_snapshot is None:
                    example_snapshot = {
                        "cos_map": cos_map_t.copy(),
                        "central_mask": central_t.copy(),
                    }

            if step < n_steps:
                Bx_t, By_t = one_step(
                    Bx_t, By_t, Bx_target, seed_pattern, rng,
                    dt=dt, dx=dx, dy=dy,
                    nu=nu, alpha=alpha, beta=beta, gamma=gamma,
                    noise_amp=noise_amp, noise_sigma=noise_sigma
                )

        critical_steps.append(int(np.argmax(np.gradient(failed_fraction_trace))))

    return {
        "mean_pocket_count": float(np.nanmean(final_pocket_counts)),
        "mean_width_std": float(np.nanmean(final_width_stds)),
        "mean_constraint_energy": float(np.nanmean(final_constraint_energies)),
        "mean_failed_fraction": float(np.nanmean(final_failed_fractions)),
        "mean_critical_step": float(np.nanmean(critical_steps)),
        "example_snapshot": example_snapshot,
    }

## 4. Run the compact scaling sweep

We store both raw parameters and derived dimensionless numbers.

In [ ]:
records = []

for delta in delta_vals:
    for gamma in gamma_vals:
        for nu in nu_vals:
            for beta in beta_vals:
                summary = run_parameter_point(delta=delta, nu=nu, beta=beta, gamma=gamma, n_runs=n_runs_per_point)

                Pi1 = beta / nu
                Pi2 = beta / gamma
                Pi3 = beta * (delta**2) / nu

                records.append({
                    "delta": float(delta),
                    "nu": float(nu),
                    "beta": float(beta),
                    "gamma": float(gamma),
                    "Pi1": float(Pi1),
                    "Pi2": float(Pi2),
                    "Pi3": float(Pi3),
                    "pocket_count": summary["mean_pocket_count"],
                    "width_std": summary["mean_width_std"],
                    "constraint_energy": summary["mean_constraint_energy"],
                    "failed_fraction": summary["mean_failed_fraction"],
                    "critical_step": summary["mean_critical_step"],
                    "example_snapshot": summary["example_snapshot"],
                })

print("number of parameter records =", len(records))
records[:2]

## 5. Pull arrays for plotting

In [ ]:
Pi1 = np.array([r["Pi1"] for r in records])
Pi2 = np.array([r["Pi2"] for r in records])
Pi3 = np.array([r["Pi3"] for r in records])

delta_arr = np.array([r["delta"] for r in records])
nu_arr = np.array([r["nu"] for r in records])
beta_arr = np.array([r["beta"] for r in records])
gamma_arr = np.array([r["gamma"] for r in records])

pocket_arr = np.array([r["pocket_count"] for r in records])
width_arr = np.array([r["width_std"] for r in records])
energy_arr = np.array([r["constraint_energy"] for r in records])
failed_arr = np.array([r["failed_fraction"] for r in records])
critical_arr = np.array([r["critical_step"] for r in records])

print("Pi1 range:", Pi1.min(), Pi1.max())
print("Pi2 range:", Pi2.min(), Pi2.max())
print("Pi3 range:", Pi3.min(), Pi3.max())

## 6. Primary scaling plots

These are the core dimensionless-bridge figures.

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(Pi1, pocket_arr)
plt.xlabel(r"$\Pi_1 = \beta / \nu$")
plt.ylabel("mean pocket count")
plt.title("Pocket Count vs Activity-to-Diffusion Ratio")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(Pi3, energy_arr)
plt.xlabel(r"$\Pi_3 = \beta \delta^2 / \nu$")
plt.ylabel("mean constraint energy")
plt.title("Constraint Energy vs Thickness-Aware Drive Ratio")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(Pi2, width_arr)
plt.xlabel(r"$\Pi_2 = \beta / \gamma$")
plt.ylabel("mean width std")
plt.title("Width Variation vs Activity-to-Damping Ratio")
plt.show()

## 7. Color-coded collapse checks

These test whether the derived control numbers organize the data more cleanly than raw parameters.

In [ ]:
plt.figure(figsize=(8, 5))
sc = plt.scatter(Pi3, pocket_arr, c=gamma_arr)
plt.xlabel(r"$\Pi_3 = \beta \delta^2 / \nu$")
plt.ylabel("mean pocket count")
plt.title("Pocket Count vs Thickness-Aware Drive Ratio (colored by gamma)")
plt.colorbar(sc, label="gamma")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sc = plt.scatter(Pi3, energy_arr, c=delta_arr)
plt.xlabel(r"$\Pi_3 = \beta \delta^2 / \nu$")
plt.ylabel("mean constraint energy")
plt.title("Constraint Energy vs Thickness-Aware Drive Ratio (colored by delta)")
plt.colorbar(sc, label="delta")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sc = plt.scatter(Pi1, width_arr, c=nu_arr)
plt.xlabel(r"$\Pi_1 = \beta / \nu$")
plt.ylabel("mean width std")
plt.title("Width Variation vs Activity-to-Diffusion Ratio (colored by nu)")
plt.colorbar(sc, label="nu")
plt.show()

## 8. Raw-parameter comparison plots

These are included for contrast against the dimensionless organization.

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(beta_arr, pocket_arr)
plt.xlabel("beta")
plt.ylabel("mean pocket count")
plt.title("Pocket Count vs Raw Activity Strength")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(nu_arr, energy_arr)
plt.xlabel("nu")
plt.ylabel("mean constraint energy")
plt.title("Constraint Energy vs Raw Diffusion")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(gamma_arr, width_arr)
plt.xlabel("gamma")
plt.ylabel("mean width std")
plt.title("Width Variation vs Raw Damping")
plt.show()

## 9. Simple regime labeling

We create a coarse visual label based on pocket count to mark three regimes:

- smooth
- bounded fragmentation
- stronger drive

In [ ]:
labels = []
for p in pocket_arr:
    if p < 2.0:
        labels.append("smooth")
    elif p < 5.0:
        labels.append("bounded")
    else:
        labels.append("strong")

labels = np.array(labels)

for lab in ["smooth", "bounded", "strong"]:
    idx = labels == lab
    plt.figure(figsize=(8, 5))
    plt.scatter(Pi3[idx], energy_arr[idx])
    plt.xlabel(r"$\Pi_3 = \beta \delta^2 / \nu$")
    plt.ylabel("mean constraint energy")
    plt.title(f"{lab.capitalize()} Regime Points")
    plt.show()

## 10. Representative snapshots

We show one representative point from each coarse regime.

In [ ]:
def pick_representative(target_label):
    idx = np.where(labels == target_label)[0]
    if len(idx) == 0:
        return None
    # choose middle point in sorted Pi3 for stability
    local = idx[np.argsort(Pi3[idx])]
    return local[len(local)//2]

rep_indices = {
    "smooth": pick_representative("smooth"),
    "bounded": pick_representative("bounded"),
    "strong": pick_representative("strong"),
}

rep_indices

In [ ]:
for name, idx in rep_indices.items():
    if idx is None:
        continue
    rec = records[idx]
    snap = rec["example_snapshot"]

    plt.figure(figsize=(9, 4.6))
    plt.imshow(
        snap["cos_map"],
        extent=[x.min(), x.max(), y.min(), y.max()],
        origin="lower",
        aspect="auto"
    )
    plt.colorbar(label="cross-sheet cosine")
    plt.contour(
        X, Y, snap["cos_map"],
        levels=[THRESHOLD],
        colors="white",
        linewidths=1.8,
        linestyles="--"
    )
    plt.xlabel("x")
    plt.ylabel("y")
    plt.title(
        f"{name} regime cosine map | "
        f"delta={rec['delta']}, nu={rec['nu']}, beta={rec['beta']}, gamma={rec['gamma']}"
    )
    plt.show()

    plt.figure(figsize=(9, 4.6))
    plt.imshow(
        snap["central_mask"],
        extent=[x.min(), x.max(), y.min(), y.max()],
        origin="lower",
        aspect="auto"
    )
    plt.xlabel("x")
    plt.ylabel("y")
    plt.title(f"{name} regime central failed region")
    plt.show()

## 11. Compact dimensionless summary table

This table is useful for README and paper drafting.

In [ ]:
for r in records:
    print(
        f"delta={r['delta']:.2f} | nu={r['nu']:.4f} | beta={r['beta']:.2f} | gamma={r['gamma']:.2f} | "
        f"Pi1={r['Pi1']:.2f} | Pi2={r['Pi2']:.2f} | Pi3={r['Pi3']:.4f} | "
        f"pockets={r['pocket_count']:.3f} | width_std={r['width_std']:.4f} | "
        f"energy={r['constraint_energy']:.4f} | failed_frac={r['failed_fraction']:.4f} | "
        f"critical_step={r['critical_step']:.2f}"
    )

## 12. Interpretation

Notebook 07 adds a genuine physics-facing layer to the repo.

What it supports:

> Fragmentation onset and persistence are better organized by timescale-ratio variables than by raw parameters alone.

Most important derived controls:

- \(\Pi_1 = eta/
u\): drive vs smoothing
- \(\Pi_2 = eta/\gamma\): drive vs damping
- \(\Pi_3 = eta \delta^2 / 
u\): thickness-aware drive ratio

What the notebook is trying to show:

- increasing \(\Pi_1\) tends to favor more pocketed structure
- increasing \(\Pi_3\) tends to raise constraint energy and fragmentation intensity
- \(\Pi_2\) helps organize persistence of transverse corrugation

What it does **not** claim:

- exact Lundquist-number equivalence
- exact Sweet–Parker or plasmoid-scaling reproduction
- calibrated solar-plasma reconnection rates

Best interpretation:

> a dimensionless bridge from the geometric precursor model to familiar reconnection-style control logic.